In [10]:
import cv2
from ultralytics import YOLO

In [25]:
# 1. Load your best model
model = YOLO('../best_yolov8_coral_reef/runs/detect/reef_coral/weights/best.pt')

# 2. Setup Video
video_path = 'Match 8 (R2) - 2025 Iowa Regional.mp4'
cap = cv2.VideoCapture(video_path)

In [26]:
OUTPUT_PATH = 'debug_scoring_trimmed.mp4'
NUM_SLOTS = 6 

# Get video properties
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Calculate the Cutoff (Total Frames - (49 seconds * FPS))
# This ensures it stops exactly 49 seconds before the actual file ends
cutoff_frame = total_frames - int(49 * fps)

crop_h = int(height * (2/5))
start_y = height - crop_h

# Setup Output
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, crop_h))

reef_grids = [[0] * NUM_SLOTS for _ in range(5)]
claimed_coral_ids = set()
total_points = 0
current_frame_idx = 0

print(f"Total Frames: {total_frames} | Stopping at Frame: {cutoff_frame}")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or current_frame_idx >= cutoff_frame:
        print(f"Reached cutoff point at frame {current_frame_idx}. Stopping.")
        break

    # --- SCORING & VISUAL LOGIC ---
    cropped = frame[start_y:height, 0:width]
    results = model.track(cropped, persist=True, tracker="bytetrack.yaml", conf=0.3, verbose=False)
    annotated_frame = results[0].plot()

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        clss = results[0].boxes.cls.cpu().numpy().astype(int)
        ids = results[0].boxes.id.cpu().numpy().astype(int)

        frame_reefs = [boxes[i] for i, c in enumerate(clss) if model.names[c] == 'reef']
        frame_reefs.sort(key=lambda x: x[0])

        for r_idx, r_box in enumerate(frame_reefs):
            rx1, ry1, rx2, ry2 = r_box.astype(int)
            scoring_limit = int(ry1 + ((ry2 - ry1) * 0.20))
            
            # Draw visual zones
            overlay = annotated_frame.copy()
            cv2.rectangle(overlay, (rx1, ry1), (rx2, scoring_limit), (0, 255, 0), -1)
            cv2.addWeighted(overlay, 0.3, annotated_frame, 0.7, 0, annotated_frame)

        for i, c_id in enumerate(ids):
            if model.names[clss[i]] == 'coral' and c_id not in claimed_coral_ids:
                cx, cy = (boxes[i][0] + boxes[i][2])/2, (boxes[i][1] + boxes[i][3])/2
                
                for r_idx, r_box in enumerate(frame_reefs):
                    rx1, ry1, rx2, ry2 = r_box
                    if (rx1 < cx < rx2) and (ry1 < cy < ry1 + (ry2-ry1)*0.20):
                        rel_x = cx - rx1
                        slot_idx = int((rel_x / (rx2-rx1)) * NUM_SLOTS)
                        slot_idx = max(0, min(slot_idx, NUM_SLOTS - 1))
                        
                        if r_idx < len(reef_grids) and reef_grids[r_idx][slot_idx] < 2:
                            reef_grids[r_idx][slot_idx] += 1
                            claimed_coral_ids.add(c_id)
                            total_points += 4
                            cv2.circle(annotated_frame, (int(cx), int(cy)), 20, (0, 255, 255), -1)

    cv2.putText(annotated_frame, f"SCORE: {total_points}", (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

    out.write(annotated_frame)
    current_frame_idx += 1

cap.release()
out.release()
print(f"Analysis Complete. Total Score up to the 49s-remaining mark: {total_points}")

Total Frames: 5674 | Stopping at Frame: 4206
Reached cutoff point at frame 4206. Stopping.
Analysis Complete. Total Score up to the 49s-remaining mark: 92


In [27]:
print("\n" + "="*30)
print("FINAL REEF OCCUPANCY REPORT")
print("="*30)

for r_idx, slots in enumerate(reef_grids):
    # Only print reefs that actually had at least one coral detected
    if sum(slots) > 0:
        # Create a visual string: [0, 2, 1, 0, 0, 0] -> "Slot 0: 0 | Slot 1: 2 | Slot 2: 1 ..."
        status_str = " | ".join([f"Slot {i}: {val}" for i, val in enumerate(slots)])
        print(f"REEF {r_idx}: {status_str}")
        
        # Optional: Print total corals on this specific reef
        print(f"   -> Total Corals on Reef {r_idx}: {sum(slots)}")
        print("-" * 30)

print(f"GRAND TOTAL SCORE: {total_points} points")
print("="*30)


FINAL REEF OCCUPANCY REPORT
REEF 0: Slot 0: 2 | Slot 1: 2 | Slot 2: 2 | Slot 3: 2 | Slot 4: 2 | Slot 5: 2
   -> Total Corals on Reef 0: 12
------------------------------
REEF 1: Slot 0: 2 | Slot 1: 2 | Slot 2: 2 | Slot 3: 2 | Slot 4: 1 | Slot 5: 2
   -> Total Corals on Reef 1: 11
------------------------------
GRAND TOTAL SCORE: 92 points
